# PointNet++ Inference & Testing Only
This notebook is lightweight and designed purely for testing your pre-trained weights on new `.pcd` point clouds! It does not download the massive dataset, saving you a lot of time and disk space.

In [ ]:
# 1. Mount Google Drive to access your saved weights
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Setup PointNet++ Repository and Install Dependencies
import os
if not os.path.exists('/content/Pointnet_Pointnet2_pytorch'):
    !git clone https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git
    
!pip install -q pypcd4 pandas matplotlib tqdm

In [ ]:
%%writefile outdoor_pointnet.py
import os
import sys
import torch
import torch.nn as nn

repo_path = os.path.abspath('Pointnet_Pointnet2_pytorch')
if repo_path not in sys.path:
    sys.path.append(repo_path)

import models.pointnet2_sem_seg as pointnet2_sem_seg
from models.pointnet2_utils import PointNetSetAbstraction

def get_outdoor_model(num_classes=20, input_channels=1):
    model = pointnet2_sem_seg.get_model(num_classes)
    
    # Note: input_channels+6 because the repo passes all 3 coordinates PLUS the 4 feature channels into sa1.
    model.sa1 = PointNetSetAbstraction(1024, 0.1, 32, input_channels + 6, [32, 32, 64], False)
    model.conv2 = nn.Conv1d(128, num_classes, 1)
    return model

def load_pretrained_weights(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    
    # Handle both wrapped dictionaries (GitHub) and direct state_dicts (our training script)
    if 'model_state_dict' in checkpoint:
        pretrained_dict = checkpoint['model_state_dict']
    else:
        pretrained_dict = checkpoint
        
    model_dict = model.state_dict()
    
    filtered_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)
    return model


## 3. Run Inference on a `.pcd` File
Upload any `.pcd` file to Colab's left sidebar, change the filename at the very bottom of this block, and run the cell to visualize the 3D semantic segmentation!

In [ ]:
%matplotlib inline
from pypcd4 import PointCloud
import numpy as np
import torch
import matplotlib.pyplot as plt
from outdoor_pointnet import get_outdoor_model, load_pretrained_weights

def get_semantickitti_colors():
    return {
        0: [255, 255, 255],   1: [100, 150, 245],   2: [100, 230, 245],   3: [30, 60, 150],     
        4: [80, 30, 180],     5: [100, 80, 250],    6: [255, 30, 30],     7: [255, 40, 200],    
        8: [150, 30, 90],     9: [255, 0, 255],     10: [255, 150, 255],  11: [75, 0, 75],      
        12: [175, 0, 75],     13: [255, 200, 0],    14: [255, 120, 50],   15: [0, 175, 0],      
        16: [135, 60, 0],     17: [150, 240, 80],   18: [255, 240, 150],  19: [255, 0, 0]
    }

def get_semantickitti_names():
    return {
        0: 'unlabeled', 1: 'car', 2: 'bicycle', 3: 'motorcycle', 4: 'truck',
        5: 'other-vehicle', 6: 'person', 7: 'bicyclist', 8: 'motorcyclist',
        9: 'road', 10: 'parking', 11: 'sidewalk', 12: 'other-ground', 13: 'building',
        14: 'fence', 15: 'vegetation', 16: 'trunk', 17: 'terrain', 18: 'pole', 19: 'traffic-sign'
    }

def test_on_pcd(pcd_path):
    if not os.path.exists(pcd_path):
        print(f"ERROR: Could not find {pcd_path}. Please upload it to Colab.")
        return
        
    print(f"Loading custom point cloud from {pcd_path}...")
    pc = PointCloud.from_path(pcd_path)
    
    try:
        scan = pc.numpy(['x', 'y', 'z', 'intensity'])
    except Exception as e:
        print(f"Warning: Could not find 'intensity' field. Defaulting to 0 intensity.")
        xyz = pc.numpy(['x', 'y', 'z'])
        scan = np.column_stack((xyz, np.zeros((xyz.shape[0], 1))))
    
    scan = scan.astype(np.float32)
    scan = scan[~np.isnan(scan).any(axis=1)]
    scan = scan[np.isfinite(scan).all(axis=1)]
    
    # Zero-center the geometry to prevent PyTorch float32 precision crash in PointNet++ query_ball_point!
    centroid = np.mean(scan[:, :3], axis=0)
    scan[:, :3] = scan[:, :3] - centroid
    
    if np.max(scan[:, 3]) > 1.0:
        scan[:, 3] = scan[:, 3] / 255.0
    
    num_raw = scan.shape[0]
    if num_raw >= 4096:
        choice = np.random.choice(num_raw, 4096, replace=False)
    else:
        choice = np.random.choice(num_raw, 4096, replace=True)
    sampled_scan = scan[choice, :]
    
    point_features = torch.tensor(sampled_scan, dtype=torch.float32).transpose(0, 1)
    inputs = point_features.unsqueeze(0)
    xyz = inputs[0, :3, :].transpose(0, 1).numpy()
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_outdoor_model(num_classes=20, input_channels=1)
    
    # --- USE NEW MATLAB WEIGHTS ---
    CHECKPOINT_PATH = '/content/drive/MyDrive/checkpoints_matlab/matlab_epoch_10.pth'
    
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Loading weights from {CHECKPOINT_PATH}...")
        model = load_pretrained_weights(model, CHECKPOINT_PATH)
    else:
        print(f"ERROR: Checkpoint not found at {CHECKPOINT_PATH}!")
        return
        
    model = model.to(device)
    model.eval()
    
    print("Running inference...")
    inputs = inputs.to(device)
    with torch.no_grad():
        predictions, _ = model(inputs)
    
    pred_labels = torch.argmax(predictions, dim=2).squeeze(0).cpu().numpy()
    
    name_map = get_semantickitti_names()
    unique_classes = np.unique(pred_labels)
    detected_objects = [name_map.get(c, "Unknown") for c in unique_classes if c != 0]
    
    if len(detected_objects) == 0:
        print(f"\n---> Warning: PointNet++ predicted 'unlabeled' for every single point in the frame!")
    else:
        print(f"\n---> Objects detected by PointNet++ in this custom frame: {', '.join(detected_objects)}\n")
    
    color_map = get_semantickitti_colors()
    pred_colors = np.array([color_map.get(l, [255, 255, 255]) for l in pred_labels]) / 255.0
    
    plt.figure(figsize=(10, 10))
    plt.scatter(xyz[:, 0], xyz[:, 1], c=pred_colors, s=5, alpha=0.8)
    plt.title("PointNet++ MATLAB Finetuned Prediction", fontsize=16)
    plt.axis('equal')
    plt.gca().set_facecolor('black')
    plt.show()

# Simply change 'sample.pcd' to the name of any file you upload to Colab!
test_on_pcd('/content/frame_0002.pcd')
